# Siamese network

<img src='illustration/siamese.JPG'>

In [1]:
import os
os.environ['KERAS_BACKEND'] = 'torch'
from typing import Union, Tuple

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import keras
from keras.layers import Input, Lambda, Dense, Flatten, MaxPool2D, Conv2D, GlobalAvgPool2D

from sklearn.metrics import confusion_matrix, classification_report

In [5]:
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
IMG_SHAPE = (28, 28, 1)
np.random.seed(1)

In [7]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 28, 28, 1)/255
X_test = X_test.reshape(-1, 28, 28, 1)/255

(60000, 28, 28, 1)

In [10]:
def get_pair(X:np.array, y:np.array, coef=0.5) -> Union[np.array, np.array]:
    """
    This function creates pairs of images. Some belonging to the same class
    depending the ratio.
    
    Args:
        X (np.array): Image used to create the pairs
        y (np.array): class of each sample
        coef (float, optional): ratio of non pairs. Defaults to 0.5.

    Returns:
        tuple (np.array, np.array): X_pair with the pairs and y with 0 and 1 for each sample.
        1 => images are similars (i.e. they are true pairs, belongs to same class). The concept of
        similarity hihgly depends on the loss used. Metrics like euclidean distance does not lead to get an embedding
        of images belonging to the same classes. This might leads to biases and abnormalities. Please refers to other
        loss such as contrastive loss, Semi hard loss, Triplet loss, etc.
        0 => images are not similars
    """
    X_pair = np.empty(X.shape)
    y_pair = np.ones(y.shape, dtype=np.uint8)

    for i in range(10):
        n_samples = y[y == i].shape[0]

        idx = np.random.choice(np.where(y == i)[0], n_samples)
        X_pair[y == i] = X[idx]

        size = int(n_samples*coef)

        idx = np.random.choice(np.where(y == i)[0], size, replace=False)

        idx_false = np.random.choice(np.where(y != i)[0], size, replace=False)
        X_pair[idx] = X[idx_false]

        y_pair[idx] = 0

    return X_pair, y_pair

def show_sample(X:np.array, X_pair:np.array, y_pair:np.array, n_samples=10) -> None:
    # TODO: subplot the subplots
    """
    Show some pair samples

    Args:
        X (np.array): image
        X_pair (np.array): pair images
        y_pair (np.array): class of the pair are they similar or not
        n_samples (int, optional): number of sample to display. Defaults to 10.
    """
    for i in np.random.randint(0, X.shape[0], n_samples):
        plt.subplot(1, 2, 1)
        plt.imshow(X[i], cmap='gray')

        plt.subplot(1, 2, 2)
        plt.imshow(X_pair[i], cmap='gray')

        plt.suptitle(y_pair[i])

        plt.show()


In [12]:
X_train_pair, y_train_pair = get_pair(X_train, y_train)
X_test_pair, y_test_pair = get_pair(X_test, y_test)

In [13]:
print(X_train.shape)
X_train_pair.shape

(60000, 28, 28, 1)


(60000, 28, 28, 1)

In [15]:
# show_sample(X_train, X_train_pair, y_train_pair)

In [18]:
def get_cnn(initializer):
    inputs = Input(IMG_SHAPE)

    x = Conv2D(64, 4, activation='relu',
               kernel_initializer=initializer)(inputs)
    x = MaxPool2D()(x)

    x = Conv2D(64, 4, activation='relu', kernel_initializer=initializer)(x)
    x = MaxPool2D()(x)

    x = Flatten()(x)
    # x = GlobalAvgPool2D()(x)
    outputs = Dense(128, activation='relu', kernel_initializer=initializer)(x)

    cnn = keras.Model(inputs, outputs)
    return cnn


def euclidean_distance(tensors):
    left, right = tensors
    distance = keras.ops.maximum(
        keras.ops.norm(left-right, keepdims=True, axis=1), 1e-10)  # 1* 10^-18
    return distance


def contrastive_loss(y_true, y_pred, margin=1.0):
    y_true = keras.ops.cast(y_true, y_pred.dtype)
    return y_true * keras.ops.square(y_pred) + (1.0 - y_true) * keras.ops.square(keras.ops.maximum(margin - y_pred, 0.0))


def get_siamese(initializer='glorot_uniform'):
    cnn = get_cnn(initializer=initializer)

    img_left = Input(IMG_SHAPE)
    img_right = Input(IMG_SHAPE)

    left = cnn(img_left)
    right = cnn(img_right)

    distance = Lambda(euclidean_distance)((left, right))

    # labels = Dense(2, activation='softmax', name='labels')(distance)
    # distance = Dense(1, activation='linear', name='distance')(distance)

    siamese = keras.Model([img_left, img_right], outputs=distance)

    return siamese


In [19]:
keras.utils.plot_model(get_siamese(),
                       to_file='model_constrastive_loss.png',
                       expand_nested=True,
                       show_shapes=True)


('You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) ', 'for plot_model/model_to_dot to work.')


In [ ]:
siamese = get_siamese('glorot_uniform')

siamese.compile(loss=contrastive_loss,
                optimizer=keras.optimizers.Adam(learning_rate=0.001),
                metrics=['accuracy'])

hsitory = siamese.fit([X_train, X_train_pair], y_train_pair,
                      validation_data=([X_test, X_test_pair], y_test_pair),
                      epochs=2)


Epoch 1/2
1875/1875 [==============================] - 98s 52ms/step - loss: 0.1065 - distance_loss: 0.1065 - labels_accuracy: 0.4996 - val_loss: 0.0364 - val_distance_loss: 0.0364 - val_labels_accuracy: 0.4994
Epoch 2/2
1875/1875 [==============================] - 94s 50ms/step - loss: 0.0318 - distance_loss: 0.0318 - labels_accuracy: 0.4999 - val_loss: 0.0277 - val_distance_loss: 0.0277 - val_labels_accuracy: 0.4997


In [ ]:
y_pred = siamese.predict([X_test, X_test_pair])

y_pred = np.round(y_pred)
y_pred = y_pred.astype(int).reshape(-1)

print(classification_report(y_test_pair, y_pred))

plt.figure(figsize=(10, 5))
sns.heatmap(confusion_matrix(y_test_pair, y_pred),
            annot=True, cbar=False, fmt='d', cmap='Blues')
plt.show()

errors = np.where(y_test_pair != y_pred)[0]
print(errors.shape)

X = X_test[errors]
X_pair = X_test_pair[errors]
y_pair = y_test_pair[errors]


Error: Session cannot generate requests

In [ ]:
for i in np.random.randint(0, X.shape[0], 10):
    plt.subplot(1, 2, 1)
    plt.title(y_test[errors][i])
    plt.imshow(X[i], cmap='gray')

    plt.subplot(1, 2, 2)
    plt.imshow(X_pair[i], cmap='gray')

    plt.suptitle('y_pred:%s\ny_true:%s'%(y_pred[errors][i], y_pair[i]))

    plt.show()